In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv(r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed\listings_clean.csv")

# Keep priced listings without extreme prices
model_data = df[(df["is_priced"] == True) & (df["price_outlier"] == False)]

# Keep only the columns we need and drop rows with missing values
columns = ["price_num", "accommodates", "bedrooms", "room_type_group", "neighbourhood", "latitude", "longitude"]
model_data = model_data[columns].dropna()
print("Rows for the model:", len(model_data))

# Inputs (X) and target (y)
X = pd.get_dummies(model_data[["accommodates", "bedrooms", "room_type_group", "neighbourhood", "latitude", "longitude"]])
y = np.log(model_data["price_num"])

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

Rows for the model: 5401
(4320, 29) (1081, 29)


In [1]:
%pip install scikit-learn

  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.2 MB 4.2 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.2 MB 4.6 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.2 MB 4.9 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 5.2 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.2 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 5.8 MB/s  0:00:01
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- -----------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

base = DummyRegressor(strategy="mean")
base.fit(X_train, y_train)
pred_base = base.predict(X_test)

print("Baseline  MAE:", round(mean_absolute_error(np.exp(y_test), np.exp(pred_base))), "| R2:", round(r2_score(y_test, pred_base), 2))

Baseline  MAE: 127 | R2: -0.0


In [4]:
linear = LinearRegression()
linear.fit(X_train, y_train)
pred_linear = linear.predict(X_test)

print("Linear    MAE:", round(mean_absolute_error(np.exp(y_test), np.exp(pred_linear))), "| R2:", round(r2_score(y_test, pred_linear), 2))

Linear    MAE: 96 | R2: 0.4


In [5]:
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
forest.fit(X_train, y_train)
pred_forest = forest.predict(X_test)

print("Forest    MAE:", round(mean_absolute_error(np.exp(y_test), np.exp(pred_forest))), "| R2:", round(r2_score(y_test, pred_forest), 2))

Forest    MAE: 92 | R2: 0.47


In [6]:
importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(8).round(3))

bedrooms                                 0.294
longitude                                0.254
latitude                                 0.221
room_type_group_Entire home/apt          0.102
accommodates                             0.074
neighbourhood_De Pijp - Rivierenbuurt    0.013
room_type_group_Private room             0.009
neighbourhood_Centrum-West               0.008
dtype: float64


In [7]:
from sklearn.model_selection import cross_val_predict

# Predict every listing using models that never saw that listing
pred_all = cross_val_predict(forest, X, y, cv=5)

scored = model_data.copy()
scored["listing_id"] = df.loc[scored.index, "listing_id"]
scored["nights"] = df.loc[scored.index, "estimated_occupancy_l365d"]
scored["fair_price"] = np.exp(pred_all)
scored["gap_pct"] = (scored["price_num"] / scored["fair_price"] - 1) * 100

underpriced = scored[(scored["gap_pct"] <= -20) & (scored["nights"] >= 200)]
print("ML underpriced candidates:", len(underpriced), "of", len(scored))
print(underpriced[["listing_id", "price_num", "fair_price", "nights"]].head(5))

ML underpriced candidates: 278 of 5401
    listing_id  price_num  fair_price  nights
7        50523     154.06  374.965564     255
27      245927     459.00  612.404931     255
31      255809     262.47  405.183625     255
39      304082     125.53  324.668103     255
60      450767     456.67  644.591663     255


In [8]:
# 1) Median gap among the ML candidates
print("Median gap:", round(underpriced["gap_pct"].median(), 1), "%")

# 2) Compare with the SQL-style rule (peer average) on the same 5,401 listings
scored["peer_avg"] = scored.groupby(["neighbourhood", "room_type_group"])["price_num"].transform("mean")
scored["sql_rule"] = (scored["price_num"] < 0.8 * scored["peer_avg"]) & (scored["nights"] >= 200)
scored["ml_rule"] = (scored["gap_pct"] <= -20) & (scored["nights"] >= 200)
print(pd.crosstab(scored["sql_rule"], scored["ml_rule"]))

# 3) Save the ML candidate list for Power BI
out = r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed"
scored.to_csv(out + r"\ml_scored_listings.csv", index=False)

Median gap: -34.2 %
ml_rule   False  True 
sql_rule              
False      4987     48
True        136    230


In [1]:
folder = r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed"

scored["ml_underpriced"] = (scored["gap_pct"] <= -20) & (scored["nights"] >= 200)
scored.to_csv(folder + r"\ml_scored_listings.csv", index=False)

print(scored.shape)
print(scored["ml_underpriced"].sum())

NameError: name 'scored' is not defined

In [ ]:
folder = r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed"

scored["ml_underpriced"] = (scored["gap_pct"] <= -20) & (scored["nights"] >= 200)
scored.to_csv(folder + r"\ml_scored_listings.csv", index=False)

print(scored.shape)
print(scored["ml_underpriced"].sum())   